# 02 - Data Exploration

The observed record for the Ayase and Naka rivers, and the synthetic habitat
labels built on it.

Source: 環境省 水環境総合情報サイト / 埼玉県 公共用水域水質測定データ, FY2022-24.
**Twelve grab samples per station per year** - not a continuous series. See
`docs/DATA_SOURCES.md` for why no higher-frequency source exists.


In [ ]:
import sys; sys.path.insert(0, '../src')
import glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from aquanexus.data.loader import load_many, filter_stations, apply_censoring
from aquanexus.data.preprocessor import add_derived_features, observed_ranges
from aquanexus.data.synthetic import (habitat_suitability, classify,
                                      falsification_report, COLDWATER)
from aquanexus.data.validator import validate_observations

pd.set_option('display.width', 200)


## 1. Load


In [ ]:
files = sorted(glob.glob('../data/raw/waterquality/saitama_*.xlsx'))
raw = load_many(files)
print(f'{len(raw):,} samples | {raw.station.nunique()} stations | '
      f'{raw.timestamp.min().date()} to {raw.timestamp.max().date()}')

ayase = filter_stations(raw, water_body='綾瀬川')
print(f'Ayase: {len(ayase)} samples across {ayase.station.nunique()} stations')
ayase.groupby('station').size()


### Censored values

Around 9,400 values a year are flagged `<`: the reported number is the detection
limit, not a measurement. Treating those as measurements biases every statistic
upward, so the loader keeps them as published and records the qualifier
separately. `apply_censoring()` applies a policy as a deliberate step.


In [ ]:
flags = [c for c in raw.columns if c.endswith('_flag')]
counts = sorted(((raw[c].notna().sum(), c[:-5]) for c in flags), reverse=True)
print(f'{sum(n for n, _ in counts):,} censored values across {len(flags)} parameters')
for n, name in counts[:5]:
    print(f'  {n:5d}  {name}')


## 2. Quality checks


In [ ]:
print(validate_observations(ayase, required=['water_temp', 'dissolved_oxygen']))


## 3. The observed gradient

This is the relationship the habitat model must reproduce - and it is real
measured data, independent of anything generated.


In [ ]:
features = add_derived_features(ayase)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(features.water_temp, features.dissolved_oxygen, s=18, alpha=0.7)
axes[0].plot(np.sort(features.water_temp),
             features.do_saturation[np.argsort(features.water_temp)], 'r-',
             label='saturation')
axes[0].set_xlabel('water temperature (°C)'); axes[0].set_ylabel('DO (mg/L)')
axes[0].legend()

axes[1].scatter(features.water_temp, features.do_saturation_pct, s=18, alpha=0.7)
axes[1].axhline(100, color='grey', ls='--')
axes[1].set_xlabel('water temperature (°C)'); axes[1].set_ylabel('% saturation')

month = features.timestamp.dt.month
axes[2].plot(month.groupby(month).first(),
             features.groupby(month).water_temp.mean(), 'o-', label='temp (°C)')
axes[2].plot(month.groupby(month).first(),
             features.groupby(month).dissolved_oxygen.mean(), 's-', label='DO (mg/L)')
axes[2].set_xlabel('month'); axes[2].legend()
plt.tight_layout(); plt.show()

warm, cool = features[features.water_temp > 25], features[features.water_temp < 15]
print(f'DO above 25 °C: {warm.dissolved_oxygen.mean():.2f} mg/L (n={len(warm)})')
print(f'DO below 15 °C: {cool.dissolved_oxygen.mean():.2f} mg/L (n={len(cool)})')


Note the points **above** the saturation line: real supersaturation from spring
algal photosynthesis, up to 17 mg/L. `do_deficit` is deliberately allowed to go
negative rather than clipped, since supersaturation is itself a eutrophication
signal.


## 4. Design space

These bounds define the simulation sweep. They replace the guessed ranges in
`ML_STRATEGY.md` §3.1 - whose discharge range of 50-500 m³/s sits entirely above
this river's 99th percentile.


In [ ]:
keys = ['water_temp', 'discharge', 'dissolved_oxygen', 'bod',
        'nitrogen_total', 'phosphorus_total', 'suspended_solids']
observed_ranges(features, columns=keys).round(3)


## 5. Synthetic labels

**HSI is generated, not measured.** A model trained on it recovers a function
written in `aquanexus.data.synthetic`, not ecology. What makes it defensible is
that it is checked against the observed gradient above.


In [ ]:
features['hsi'] = habitat_suitability(features)
features['band'] = classify(features.hsi).values

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(features.hsi, bins=25, edgecolor='white')
axes[0].set_xlabel('HSI'); axes[0].set_ylabel('samples')

sc = axes[1].scatter(features.water_temp, features.dissolved_oxygen,
                     c=features.hsi, cmap='RdYlGn', s=28, vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label='HSI')
axes[1].set_xlabel('temperature (°C)'); axes[1].set_ylabel('DO (mg/L)')

monthly = features.groupby(features.timestamp.dt.month).hsi.mean()
axes[2].plot(monthly.index, monthly.values, 'o-')
axes[2].set_xlabel('month'); axes[2].set_ylabel('mean HSI')
plt.tight_layout(); plt.show()

print(features.band.value_counts().reindex(
    ['unsuitable', 'poor', 'moderate', 'good', 'optimal']))


### Falsification

If HSI does not fall as observed conditions worsen, the label function is wrong.

Comparisons are confined to a 15-30 °C band. Without that the test is confounded
and returns the **wrong verdict**: oxygen anti-correlates with temperature, so the
most oxygen-rich samples are winter water, which scores poorly for a warmwater
guild. Unconditioned, the DO comparison ran backwards while the labels were
behaving correctly.


In [ ]:
for profile in ('lowland_warmwater', 'coldwater'):
    r = falsification_report(features, profile)
    print(f"{profile}: {'PASS' if r['passed'] else 'FAIL'}  (n in band {r['n_in_band']})")
    print(f"   DO<5 {r['hsi_low_do']:.3f} vs DO>=8 {r['hsi_high_do']:.3f}")
    print(f"   stressed {r['hsi_stressed']:.3f} vs benign {r['hsi_benign']:.3f}")
    print(f"   spearman {r['corr_hsi_do']:.3f}\n")


The `coldwater` profile is the one `ML_STRATEGY.md` §4.2 specifies - trout and
char. It rates almost the entire summer as uninhabitable, because those species
do not live in a lowland river reaching 32.5 °C. That is why the default profile
is a eurythermal cyprinid assemblage (オイカワ, コイ, フナ).


In [ ]:
comparison = pd.DataFrame({
    'lowland_warmwater': habitat_suitability(features),
    'coldwater': habitat_suitability(features, COLDWATER),
})
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(features.water_temp, comparison.lowland_warmwater, s=18,
           alpha=0.7, label='lowland warmwater')
ax.scatter(features.water_temp, comparison.coldwater, s=18,
           alpha=0.7, label='coldwater (spec)')
ax.set_xlabel('water temperature (°C)'); ax.set_ylabel('HSI'); ax.legend()
plt.show()
comparison.describe().round(3)


## Next

`03_model_training.ipynb` joins these observations to simulated hydraulics from
notebook 01 and trains the model. Depth and velocity are only available from the
HEC-RAS side - the monitoring record has neither.
